# Deep Learning Day 3: Recurrent Neural Networks & Time-Series Analysis

## The Big Picture

Remember our journey so far?

**Day 1:** You built Dense networks for the **circles dataset** - 2 input features (x, y coordinates) predicting which circle a point belongs to. You learned about neurons, activations, loss functions, optimizers, and how to spot overfitting.

**Day 2:** You discovered that Dense networks struggle with images because they **flatten pixels and lose spatial relationships**. CNNs solved this by using filters that slide across the image, detecting patterns like edges and shapes **regardless of where they appear**.

```
THE PATTERN WE'VE SEEN:

    Problem:           Dense networks lose important structure

    Day 2 Solution:    CNNs preserve SPATIAL structure
                       (nearby pixels are related)

    Today's Problem:   What about TEMPORAL structure?
                       (nearby time steps are related)
```

Consider these scenarios where **ORDER MATTERS**:
- Stock prices: $100 → $110 → $105 → ??? (predicting tomorrow's price)
- Power consumption: 5kW → 8kW → 12kW → ??? (predicting next hour)
- Sentences: "The cat sat on the ___" (predicting the next word)

**Today, we learn Recurrent Neural Networks (RNNs) and LSTMs** - architectures designed for **sequential data** where the order of inputs carries crucial information.

**New Framework:** Today we'll use **PyTorch** - the industry-standard deep learning framework used by Meta, Tesla, OpenAI, and most research labs. It gives you more control and is essential for your career!

## Learning Objectives

By the end of this lecture, you will:
1. Understand WHY sequential data needs different architectures (like CNNs for images!)
2. Know how RNNs process sequences using "memory" (the hidden state)
3. Understand why LSTMs solve the problems of simple RNNs
4. Prepare time-series data for LSTM models (windowing, normalization)
5. Build and train LSTM models in **PyTorch**
6. Write your own training loop (more control than model.fit()!)
7. Evaluate time-series predictions with appropriate metrics

## Your Journey So Far

```
DL Day 1:    Neural Networks + How They Train
             → Perceptrons, activations, forward pass, gradient descent
             ↓
DL Day 2:    CNNs for Images (Keras)
             → Spatial patterns need special architecture
             → Filters slide across space, share weights
             ↓
DL Day 3:    RNNs/LSTMs for Sequences (PyTorch) ← YOU ARE HERE
(Today!)     → Temporal patterns need special architecture
             → "Memory" flows across time, share weights
             → New framework: PyTorch!
```

**Notice the parallel:** Just as CNNs use **weight sharing across space** (same filter everywhere), RNNs use **weight sharing across time** (same operation at every time step). This is the key insight!

---

## Setup

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Scikit-learn utilities
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# For reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")
print("✅ Setup complete!")

---

# Part 1: Why Do Sequences Need Special Treatment?

## Quick Review: What We've Built So Far

Let's recall what we've learned and the **key insight** from each architecture:

```
DENSE NETWORKS (Days 1-2) - The Circles Dataset:

    Remember our circles problem?
    
        Input: [x, y]     →    2 features (coordinates)
        Output: 0 or 1    →    inner or outer circle
    
    Key properties:
    • Each feature is independent
    • ORDER DOESN'T MATTER: [x, y] works the same as [y, x]
    • All inputs processed at once
    • Every input connects to every neuron
    
    Dense networks work great when inputs are "flat" features!


CNNs (Day 2) - The MNIST Dataset:

    Remember why Dense networks struggled with images?
    
        Problem: Flattening loses spatial relationships!
                 Pixel (5,7) and pixel (5,8) are neighbors
                 but after flattening, that's lost.
    
        Solution: Conv2D slides a filter across the image
                  • Same filter everywhere → weight sharing
                  • Preserves "nearby pixels are related"
                  • Translation invariance: "7" detected anywhere
    
    CNNs preserve SPATIAL structure!
```

**Today's question:** What if the structure we need to preserve is **TEMPORAL** (across time)?

---

## The Order Problem

Consider these three scenarios:

```
SCENARIO 1: Stock Prices

Day 1: $100 → Day 2: $110 → Day 3: $105 → Day 4: ???

To predict Day 4, we need:
    • Day 3's price (the most recent)
    • The TREND (went up, then down)
    • The PATTERN (does it always drop after a rise?)

If we shuffle: $110 → $100 → $105 → ???
    Different prediction! Order matters!


SCENARIO 2: Power Consumption

Hour 1: 5kW → Hour 2: 8kW → Hour 3: 12kW → Hour 4: ???

To predict Hour 4, we need:
    • Recent values
    • Time-of-day patterns (mornings use more?)
    • Trends (increasing, decreasing, stable)


SCENARIO 3: Sentences

"The cat sat on the ___"

To predict the blank, we need:
    • Previous words ("cat", "sat", "on", "the")
    • Grammar rules (article → noun expected)
    • Context (sitting implies a surface)

If we shuffle: "sat the on cat the ___"
    Makes no sense! Order matters!
```

---

In [ ]:
# VISUAL: Why order matters - the SAME numbers, ordered vs shuffled

np.random.seed(0)
steps = np.arange(24)
ordered = 100 + steps * 1.5 + 8 * np.sin(steps / 2.0)   # trend + cycle = a clear pattern
shuffled = ordered.copy()
np.random.shuffle(shuffled)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(steps, ordered, 'o-', color='steelblue', linewidth=2)
axes[0].scatter([len(steps)], [ordered[-1] + 3], color='green', s=260, marker='*', zorder=5)
axes[0].annotate('next value?\n(easy to guess)', xy=(len(steps), ordered[-1] + 3),
                 xytext=(len(steps) - 8, ordered[-1] - 18), fontsize=11, color='green')
axes[0].set_title('ORDERED: the pattern tells you what comes next', fontsize=12)
axes[0].set_xlabel('time step'); axes[0].set_ylabel('value'); axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, shuffled, 'o-', color='indianred', linewidth=2)
axes[1].scatter([len(steps)], [shuffled[-1]], color='crimson', s=260, marker='*', zorder=5)
axes[1].annotate('next value?\n(impossible)', xy=(len(steps), shuffled[-1]),
                 xytext=(len(steps) - 8, shuffled.min()), fontsize=11, color='crimson')
axes[1].set_title('SHUFFLED: same numbers, pattern destroyed', fontsize=12)
axes[1].set_xlabel('time step'); axes[1].set_ylabel('value'); axes[1].grid(True, alpha=0.3)

plt.suptitle('Order carries the meaning: shuffle the sequence and the future becomes unpredictable',
             fontsize=13)
plt.tight_layout(); plt.show()

print("WHAT YOU'RE SEEING:")
print("  - Left: an ordered series - trend plus a cycle. You can eyeball the next point.")
print("  - Right: the EXACT same values, shuffled. The pattern is gone.")
print("  - The numbers didn't change - only their order did.")
print("  - That is 'temporal structure': for sequences, order IS the information.")

## Why Dense Networks Fail on Sequences

Let's try to use a Dense network for sequence prediction:

```
PROBLEM: Predict next value from previous 3 values

Sequence: [100, 110, 105] → ??? 


DENSE NETWORK APPROACH:

    Input: [100, 110, 105]    (3 features)
           ↓
    Dense(16, relu)
           ↓
    Dense(1)                  (prediction)


PROBLEMS:

1. FIXED INPUT SIZE
   • Must always have exactly 3 values
   • Can't handle: [100, 110, 105, 108, 112] → ???
   • Different window sizes = different models!

2. NO WEIGHT SHARING ACROSS TIME
   • Position 1 has different weights than position 2
   • "100 at position 1" treated differently than "100 at position 2"
   • Network must learn patterns separately for each position!

3. NO MEMORY MECHANISM
   • Sees [100, 110, 105] as a flat vector
   • Doesn't know 100 came BEFORE 110
   • Must infer order from position alone

4. PARAMETER EXPLOSION FOR LONG SEQUENCES
   • 100 time steps × 1 feature = 100 inputs
   • Dense(128) = 100 × 128 = 12,800 parameters
   • 1000 time steps = 128,000 parameters just for first layer!
```

**We need an architecture that:**
- Processes one time step at a time
- Maintains "memory" of what it has seen
- Uses the same weights for each time step
- Can handle variable-length sequences

**Enter: Recurrent Neural Networks!**

---

---

# Part 2: Recurrent Neural Networks (RNNs)

## The Key Idea: Memory Through Loops

Remember how CNNs solved the spatial problem?
- **Same filter** slides across different positions
- Detects patterns **regardless of where they appear**
- This is called **weight sharing across space**

RNNs use the **exact same principle for time**:
- **Same operation** applied at each time step
- Processes sequences **step by step**
- Called **weight sharing across time**

But RNNs add something CNNs don't have: **MEMORY**.

```
THE RNN INSIGHT:

    When processing step t, we need to "remember" steps 1, 2, ..., t-1
    
    How? We maintain a "hidden state" h_t that summarizes everything
    the network has seen so far.
    

SIMPLE RNN ARCHITECTURE:

For each time step t:

    1. Take input x_t (current value)
    2. Take hidden state h_{t-1} (memory from before)
    3. Compute new hidden state h_t
    4. Optionally produce output

            ┌──────────────────────────────────────┐
            │                LOOP                  │
            │    (same operation, new data)        │
            ↓                                      │
    ┌─────────────┐                               │
    │   RNN Cell  │                               │
    │             │                               │
    │  h_t = f(x_t, h_{t-1})                      │
    │             │                               │
    └──────┬──────┘                               │
           │                                       │
           ├─────────────────────────────────────→┘
           │        (h_t becomes h_{t-1}
           ↓         for next step)
        Output


UNROLLED VIEW (easier to visualize):

Time:     t=1          t=2          t=3          t=4
          ────         ────         ────         ────
Input:    x_1          x_2          x_3          x_4
           ↓            ↓            ↓            ↓
        ┌─────┐      ┌─────┐      ┌─────┐      ┌─────┐
h_0 ──→ │ RNN │─h_1─→│ RNN │─h_2─→│ RNN │─h_3─→│ RNN │──→ h_4
        └──┬──┘      └──┬──┘      └──┬──┘      └──┬──┘
           ↓            ↓            ↓            ↓
         (o_1)        (o_2)        (o_3)        (o_4)
         

KEY PARALLEL TO CNNs:

    CNN:   Same FILTER  at every POSITION → weight sharing across SPACE
    RNN:   Same WEIGHTS at every TIME     → weight sharing across TIME
    
    Both share weights, just in different dimensions!
```

---

## The Hidden State: The Network's "Memory"

The hidden state `h_t` is the key concept:

```
WHAT IS THE HIDDEN STATE?

h_t = A vector that summarizes everything the network
      has seen up to time t.

Analogy: Reading a book
    • h_t is like your "mental summary" after reading page t
    • It contains relevant information from pages 1 to t
    • When you read page t+1, you update your mental summary
    • You don't remember every word, just the important stuff


SIMPLE RNN FORMULA:

    h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b)
          ────  ────────────   ────────────   ─
           │         │              │         │
           │         │              │         └─ bias
           │         │              └─ weight × current input
           │         └─ weight × previous hidden state
           └─ activation (squishes to -1 to 1)

In words:
    "New memory = activation(combine old memory + new input)"


DIMENSIONS:

    x_t:   (input_size,)       e.g., (1,) for single value
    h_t:   (hidden_size,)      e.g., (64,) for 64 "memory cells"
    W_xh:  (input_size, hidden_size)
    W_hh:  (hidden_size, hidden_size)
    
    More hidden units = more "memory capacity"
```

---

## RNN Processing Example

Let's trace through processing a simple sequence:

```
SEQUENCE: [100, 110, 105]
TASK:     Predict next value

Step 1 (t=1): Input x_1 = 100
    • Start with h_0 = zeros (no prior memory)
    • Compute h_1 = tanh(W_hh·[0,0,...] + W_xh·[100])
    • h_1 now contains information about "100"

Step 2 (t=2): Input x_2 = 110
    • Use h_1 from previous step
    • Compute h_2 = tanh(W_hh·h_1 + W_xh·[110])
    • h_2 now contains information about "100, 110"
    • Network might encode: "started at 100, went up by 10"

Step 3 (t=3): Input x_3 = 105
    • Use h_2 from previous step
    • Compute h_3 = tanh(W_hh·h_2 + W_xh·[105])
    • h_3 now contains information about "100, 110, 105"
    • Network might encode: "started at 100, rose to 110, fell to 105"

Final:
    • Use h_3 to predict next value
    • prediction = Dense(h_3) → e.g., 108
    • The prediction incorporates the ENTIRE sequence history!
```

---

In [ ]:
# VISUAL: The hidden state = memory. Watch it update as the RNN reads each value.

rng = np.random.default_rng(1)
H = 6                                   # 6 "memory cells"
Wxh = rng.normal(0, 0.8, (H, 1))
Whh = rng.normal(0, 0.8, (H, H))
seq = np.array([100, 110, 105, 115, 120, 118]) / 120.0   # scaled inputs

h = np.zeros((H, 1))
states = []
for x in seq:
    h = np.tanh(Wxh * x + Whh @ h)      # THE recurrence: new memory from old memory + new input
    states.append(h.ravel().copy())
states = np.array(states).T             # (H, time)

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(states, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(seq)))
ax.set_xticklabels([f'x={int(v * 120)}' for v in seq])
ax.set_yticks(range(H))
ax.set_yticklabels([f'cell {i}' for i in range(H)])
ax.set_xlabel('time step (each new input updates the memory)', fontsize=12)
ax.set_title("The hidden state h: 6 memory cells, updated at every step", fontsize=13)
fig.colorbar(im, ax=ax, label='memory value (-1 to 1)')
plt.tight_layout(); plt.show()

print("WHAT YOU'RE SEEING:")
print("  - Each COLUMN is the memory vector h right after reading one input.")
print("  - Each ROW is one of the 6 memory cells, colored by its value (-1 to 1).")
print("  - The memory CHANGES with every new value, but each step is built from the one before.")
print("  - That carried-forward vector is how the network 'remembers' what it has seen.")

## The Problem with Simple RNNs: Vanishing Gradients

Simple RNNs have a fatal flaw:

```
THE PROBLEM: Long sequences lose early information

Consider: 100 time steps

    x_1 → x_2 → x_3 → ... → x_99 → x_100
     │     │     │            │       │
     └─────┴─────┴────────────┴───────┘
                    ↓
              h_100 (final hidden state)

To learn how x_1 affects h_100, gradients must flow through
99 multiplication steps during backpropagation.


WHAT HAPPENS?

If weights < 1:  gradient × weight × weight × ... → 0
                 "Vanishing gradient" - can't learn long dependencies

If weights > 1:  gradient × weight × weight × ... → ∞
                 "Exploding gradient" - training becomes unstable


ANALOGY: Telephone Game

    Person 1 whispers to Person 2, who whispers to Person 3...
    After 100 people, the original message is lost!
    
    Same with RNNs: information from x_1 gets "diluted"
    as it passes through many time steps.


PRACTICAL EFFECT:

    Simple RNNs work for ~10-20 time steps
    Beyond that, they "forget" early inputs
    This is BAD for:
        • Long documents
        • Multi-day patterns
        • Music composition
        • Complex sequences
```

**Solution: LSTM (Long Short-Term Memory)**

---

In [ ]:
# VISUAL: Why simple RNNs forget - the vanishing (and exploding) gradient

steps = np.arange(1, 51)
settings = [
    (0.8, 'weight 0.8  ->  VANISHES to 0', 'tab:blue'),
    (1.0, 'weight 1.0  ->  stays flat', 'gray'),
    (1.2, 'weight 1.2  ->  EXPLODES', 'tab:red'),
]

plt.figure(figsize=(11, 5))
for w, label, color in settings:
    plt.plot(steps, w ** steps, 'o-', color=color, markersize=3, linewidth=2, label=label)
plt.yscale('log')
plt.axhline(1.0, color='black', linewidth=0.8, alpha=0.4)
plt.xlabel('how many time steps the signal travels back', fontsize=12)
plt.ylabel('gradient size (log scale)', fontsize=12)
plt.title('Backprop multiplies the same weight once per step\n'
          'Over many steps the signal either vanishes or explodes', fontsize=13)
plt.legend(fontsize=11); plt.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print("WHAT YOU'RE SEEING:")
print("  - To learn how step 1 affects step 50, the gradient passes back through ~50 steps.")
print("  - Each step multiplies by roughly the same weight.")
print("  - Weight below 1 (blue): the signal shrinks toward zero -> the network FORGETS.")
print("  - Weight above 1 (red): the signal blows up -> training goes unstable.")
print("  - This is why a plain RNN only remembers ~10-20 steps. LSTMs fix it.")

---

# Part 3: LSTM - Long Short-Term Memory

## The Key Innovation: Gated Memory

Remember from **Day 1** when we discussed backpropagation? Gradients flow backward through the network, updating weights layer by layer. 

The vanishing gradient problem means: **gradients get smaller and smaller** as they flow backward through many steps. It's like trying to hear someone whisper through a very long telephone chain - the message fades!

**LSTMs solve this with GATES** - learned mechanisms that control information flow:

```
LSTM vs SIMPLE RNN:

Simple RNN:  h_t = tanh(W · [h_{t-1}, x_t])
             
             One operation, memory gets OVERWRITTEN each step
             Like erasing your notebook and rewriting everything
             every time you hear a new word.

LSTM:        Uses THREE gates to control:
             1. What to FORGET from old memory
             2. What to ADD to memory
             3. What to OUTPUT
             
             Memory is SELECTIVELY updated
             Like highlighting important notes, crossing out
             outdated ones, and reading only what's needed now.


THE THREE GATES:

┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   FORGET GATE: "What should I forget from my memory?"          │
│                                                                 │
│       f_t = sigmoid(W_f · [h_{t-1}, x_t] + b_f)                │
│                                                                 │
│       Output: Values between 0 and 1 for each memory cell     │
│       0 = completely forget                                    │
│       1 = completely keep                                      │
│                                                                 │
│       Example: Topic changed? Forget the old context.          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   INPUT GATE: "What new information should I add?"             │
│                                                                 │
│       i_t = sigmoid(W_i · [h_{t-1}, x_t] + b_i)                │
│       c̃_t = tanh(W_c · [h_{t-1}, x_t] + b_c)                  │
│                                                                 │
│       i_t decides HOW MUCH to add (0 to 1)                     │
│       c̃_t is WHAT to add (candidate values, -1 to 1)          │
│                                                                 │
│       Example: Important new fact? Add it. Noise? Ignore.      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   OUTPUT GATE: "What should I output from my memory?"          │
│                                                                 │
│       o_t = sigmoid(W_o · [h_{t-1}, x_t] + b_o)                │
│                                                                 │
│       Decides which parts of memory to expose                  │
│       You might store more than you currently need to share    │
│                                                                 │
│       Example: Know the whole story, but share only relevant   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘


WHY SIGMOID FOR GATES?

    Remember Day 1? Sigmoid squishes values to (0, 1)
    
    Perfect for gates:
    • 0 = "don't pass anything through"
    • 1 = "pass everything through"
    • 0.5 = "pass half through"
    
    The NETWORK LEARNS which values to forget, add, and output!
```

---

## LSTM Architecture Diagram

```
LSTM CELL (one time step):

                        c_{t-1} (cell state - long-term memory)
                           │
         ┌─────────────────┼─────────────────────────────────┐
         │                 │                                 │
         │     ┌───────────▼───────────┐                    │
         │     │    FORGET GATE        │                    │
         │     │  f_t = σ(...)         │                    │
         │     └───────────┬───────────┘                    │
         │                 │ × (multiply)                    │
         │                 ▼                                 │
         │        (f_t × c_{t-1})                           │
         │                 │                                 │
         │                 │   ┌───────────────────┐        │
         │                 │   │   INPUT GATE      │        │
         │                 │   │  i_t = σ(...)     │        │
         │                 │   │  c̃_t = tanh(...) │        │
         │                 │   └─────────┬─────────┘        │
         │                 │             │ × (multiply)      │
         │                 │             ▼                   │
         │                 │      (i_t × c̃_t)              │
         │                 │             │                   │
         │                 │      + (add)│                   │
         │                 └──────┬──────┘                   │
         │                        ▼                          │
         │                       c_t ──────────────────────────→ c_t
         │                        │   (new cell state)       │
         │                        │                          │
         │     ┌──────────────────┴──────────────────┐      │
         │     │         OUTPUT GATE                 │      │
         │     │        o_t = σ(...)                 │      │
         │     └──────────────────┬──────────────────┘      │
         │                        │                          │
         │                        ▼                          │
         │              h_t = o_t × tanh(c_t)               │
         │                        │                          │
         └────────────────────────┼──────────────────────────┘
                                  │
                                  ▼
                       h_t (hidden state - output)
                       Also passed to next time step


KEY INSIGHT: The cell state c_t is the "highway"

    • Information can flow through unchanged (if forget gate ≈ 1)
    • This allows gradients to flow easily through many steps
    • Solves the vanishing gradient problem!
```

---

In [ ]:
# VISUAL: LSTM gates (sigmoids, 0-1) and the cell-state 'highway' that beats forgetting

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: a gate is a sigmoid -> a 0..1 dial
z = np.linspace(-8, 8, 300)
gate = 1 / (1 + np.exp(-z))
axes[0].plot(z, gate, color='purple', linewidth=2.5)
axes[0].axhline(0, color='k', lw=0.8); axes[0].axhline(1, color='k', lw=0.8)
axes[0].text(-7.5, 0.06, '0 = block everything', fontsize=11, color='crimson')
axes[0].text(1.5, 0.9, '1 = let it all through', fontsize=11, color='green')
axes[0].set_title('Each gate is a sigmoid: a learned 0-to-1 dial', fontsize=12)
axes[0].set_xlabel('gate input'); axes[0].set_ylabel('how much passes'); axes[0].grid(True, alpha=0.3)

# Right: memory retention - RNN forgets, LSTM highway keeps
steps = np.arange(0, 41)
rnn_mem = 0.85 ** steps          # overwritten each step -> fades
lstm_mem = 0.997 ** steps        # cell-state highway, forget gate ~1 -> barely fades
axes[1].plot(steps, rnn_mem, 'o-', color='tab:red', markersize=3, label='Simple RNN (overwrites memory)')
axes[1].plot(steps, lstm_mem, 'o-', color='tab:green', markersize=3, label='LSTM cell-state highway')
axes[1].set_title('A fact learned at step 0: how much survives?', fontsize=12)
axes[1].set_xlabel('time steps later'); axes[1].set_ylabel('how much of the fact remains')
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

plt.suptitle('Gates decide what to keep; the cell-state highway carries it far', fontsize=13)
plt.tight_layout(); plt.show()

print("WHAT YOU'RE SEEING:")
print("  - Left: a gate is just a sigmoid - 0 blocks, 1 passes, and the network LEARNS the value.")
print("  - Right: a fact stored at step 0. The simple RNN (red) overwrites it and it fades fast.")
print("  - The LSTM (green) parks it on the cell-state 'highway' with the forget gate near 1,")
print("    so it stays almost intact 40 steps later. That is how LSTMs remember the long stuff.")

## LSTM vs Simple RNN: The Difference

```
SIMPLE RNN:

    Memory gets OVERWRITTEN every step
    h_t = tanh(W · [h_{t-1}, x_t])
    
    Like a notepad where you erase everything
    and rewrite after each observation.


LSTM:

    Memory is SELECTIVELY updated
    • Forget irrelevant old info
    • Add relevant new info
    • Output what's needed now
    
    Like a notepad where you:
    • Cross out outdated notes
    • Add new important notes
    • Read only what you need for current task


PRACTICAL DIFFERENCE:

    Simple RNN: Works for ~10-20 steps
    LSTM:       Works for 100-1000+ steps
    
    This is why LSTMs are used for:
    • Language translation (long sentences)
    • Speech recognition (long audio)
    • Time-series prediction (long patterns)
```

---

---

# Part 4: Preparing Time-Series Data for LSTM

We understand the theory. Now let's get our hands dirty!

**Good news:** Training LSTMs uses the **exact same concepts** you learned in Days 1-2:
- `model.compile()` with optimizer and loss
- `model.fit()` with epochs and batch_size
- Watching training/validation loss for overfitting

**The new part:** Data preparation is more involved because we need to create **sequences** from our time-series.

## The Windowing Approach

LSTMs need data in a specific format: **sequences of fixed length**.

Think of it like this: to predict tomorrow's weather, you look at the past 7 days. That's a "window" of 7 days sliding through time.

```
RAW TIME SERIES:

    Time:   1    2    3    4    5    6    7    8    9    10
    Value:  100  110  105  115  120  118  125  130  128  135


WINDOWED DATA (window_size=3):

    We create sequences of 3 values to predict the 4th:

    Window 1: [100, 110, 105] → 115    "Given days 1-3, predict day 4"
    Window 2: [110, 105, 115] → 120    "Given days 2-4, predict day 5"
    Window 3: [105, 115, 120] → 118    "Given days 3-5, predict day 6"
    Window 4: [115, 120, 118] → 125    "Given days 4-6, predict day 7"
    Window 5: [120, 118, 125] → 130    "Given days 5-7, predict day 8"
    Window 6: [118, 125, 130] → 128    "Given days 6-8, predict day 9"
    Window 7: [125, 130, 128] → 135    "Given days 7-9, predict day 10"

    X (input):  shape (7, 3, 1)     7 samples, 3 time steps, 1 feature
    y (target): shape (7,)         7 target values


WHY WINDOWING?

    • LSTM needs a SEQUENCE as input (not a single value)
    • Window size is a hyperparameter YOU choose
    • Larger window = more context, but slower training
    • Smaller window = less context, but faster training
    
    For daily data, common choices: 7 (week), 30 (month), 365 (year)
    For hourly data, common choices: 24 (day), 168 (week)
```

---

In [ ]:
# VISUAL: Windowing - slide a fixed window; the next point is the target

series = np.array([100, 110, 105, 115, 120, 118, 125, 130, 128, 135])
idx = np.arange(len(series))
w = 3

plt.figure(figsize=(13, 4.5))
plt.plot(idx, series, 'o-', color='gray', linewidth=1.5, zorder=1)
colors = ['tab:blue', 'tab:orange', 'tab:green']
for k, color in enumerate(colors):
    plt.axvspan(k - 0.25, k + w - 1 + 0.25, color=color, alpha=0.13)
    plt.scatter(k + w, series[k + w], color=color, s=280, marker='*', zorder=5)
    plt.annotate(f'window {k + 1} -> target',
                 xy=(k + w, series[k + w]), xytext=(k, series[k] - 14),
                 fontsize=9, color=color,
                 arrowprops=dict(arrowstyle='->', color=color))
plt.xlabel('time step', fontsize=12); plt.ylabel('value', fontsize=12)
plt.title(f'Windowing (window size {w}): each shaded block predicts the next star', fontsize=13)
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Build the sequences inline to show the 3D shape
X, y = [], []
for i in range(len(series) - w):
    X.append(series[i:i + w])
    y.append(series[i + w])
X = np.array(X).reshape(len(X), w, 1)   # (samples, time_steps, features)
y = np.array(y)

print("WHAT YOU'RE SEEING:")
print(f"  - The window is {w} steps wide. It slides one step at a time.")
print("  - Each shaded window is one INPUT; the star just after it is that window's TARGET.")
print(f"  - From {len(series)} values we made {len(X)} input-target pairs.")
print(f"  - X shape: {X.shape}  = (samples, time_steps, features)  <- the 3D shape LSTMs need")
print(f"  - y shape: {y.shape}")
print("  - The classic bug: forgetting the last '1' (features) and passing a 2D array.")

## The Shape of LSTM Input

LSTM expects a very specific input shape:

```
LSTM INPUT SHAPE: (samples, time_steps, features)

    samples:    Number of sequences (how many windows)
    time_steps: Length of each sequence (window size)
    features:   Number of variables at each time step


EXAMPLE 1: Single variable (e.g., just power consumption)

    X shape: (1000, 24, 1)
             │     │   │
             │     │   └─ 1 variable (power)
             │     └───── 24 hours of data per sample
             └───────── 1000 samples (windows)


EXAMPLE 2: Multiple variables (e.g., power + temperature + humidity)

    X shape: (1000, 24, 3)
             │     │   │
             │     │   └─ 3 variables
             │     └───── 24 hours of data per sample
             └───────── 1000 samples


COMMON MISTAKE:

    ❌ X shape: (1000, 24)      Missing the feature dimension!
    ✅ X shape: (1000, 24, 1)   Correct: explicit 1 feature

    Solution: X = X.reshape(X.shape[0], X.shape[1], 1)
    Or in Keras: layers.Reshape((time_steps, 1))
```

---

## Normalization: Why It's Critical for LSTMs

```
THE PROBLEM:

    Raw data:  [100, 110, 105, 2345, 2400, 2380, ...]
    
    • Different features have different scales
    • Large values can dominate gradients
    • LSTM activations (tanh, sigmoid) work best with small values
    • Training becomes slow or unstable


THE SOLUTION: MinMaxScaler

    Transforms all values to range [0, 1]:
    
    x_scaled = (x - x_min) / (x_max - x_min)
    
    Before: [100, 150, 50, 200]   range: 50-200
    After:  [0.33, 0.67, 0.0, 1.0]  range: 0-1


IMPORTANT RULES:

    1. FIT scaler on TRAINING data only!
       scaler.fit(X_train)
    
    2. TRANSFORM both train and test
       X_train_scaled = scaler.transform(X_train)
       X_test_scaled = scaler.transform(X_test)
    
    3. INVERSE TRANSFORM predictions for interpretability
       predictions_original = scaler.inverse_transform(predictions_scaled)

    WHY fit only on training?
    • In real use, you don't know test data statistics!
    • Prevents "data leakage" from test to train
```

---

In [ ]:
# VISUAL: Normalization - why we squeeze values into 0..1 before an LSTM

raw = np.concatenate([np.linspace(100, 150, 20), np.linspace(2300, 2400, 20)])
mn, mx = raw.min(), raw.max()
scaled = (raw - mn) / (mx - mn)         # MinMax to [0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(raw, 'o-', color='darkorange'); axes[0].set_title('RAW: values span 100 to 2400', fontsize=12)
axes[0].set_xlabel('sample'); axes[0].set_ylabel('value'); axes[0].grid(True, alpha=0.3)

axes[1].plot(scaled, 'o-', color='teal'); axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title('AFTER MinMaxScaler: everything in 0 to 1', fontsize=12)
axes[1].set_xlabel('sample'); axes[1].set_ylabel('scaled value'); axes[1].grid(True, alpha=0.3)

plt.suptitle('Normalization: same shape, friendlier scale for tanh/sigmoid inside the LSTM',
             fontsize=13)
plt.tight_layout(); plt.show()

print("WHAT YOU'RE SEEING:")
print("  - Left: raw values jump from ~100 to ~2400. Big numbers dominate the gradients.")
print("  - Right: MinMaxScaler maps them to 0..1 - the SHAPE is identical, only the scale changed.")
print("  - LSTM gates use tanh and sigmoid, which behave best on small values -> faster, stabler.")
print("  - Golden rule: FIT the scaler on training data only, then transform train and test.")

## Train/Test Split for Time-Series

Time-series data requires **different** splitting than regular data!

```
❌ WRONG: Random split (shuffling)

    Data:    [Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec]
    
    Train:   [Mar, Jun, Jan, Sep, Nov, Apr, Aug]  ← Random!
    Test:    [Feb, May, Jul, Oct, Dec]            ← Random!
    
    PROBLEM: 
    • Training on September to predict July?
    • That's using FUTURE data to predict PAST!
    • This is called "data leakage" - inflates accuracy
    • Model won't work in real predictions


✅ CORRECT: Temporal split (chronological)

    Data:    [Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec]
    
    Train:   [Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep]  ← Earlier
    Test:    [Oct, Nov, Dec]                                ← Later
    
    CORRECT:
    • Always train on PAST
    • Always test on FUTURE
    • Mimics real-world usage


IN PRACTICE:

    train_size = int(len(data) * 0.8)  # 80% for training
    train_data = data[:train_size]      # First 80%
    test_data = data[train_size:]       # Last 20%
    
    DO NOT use sklearn's train_test_split with shuffle=True!
```

---

In [ ]:
# VISUAL: Time-series split - shuffling is CHEATING; split by time instead

t = np.arange(120)
series = 50 + 0.3 * t + 10 * np.sin(t / 6.0)
rng = np.random.default_rng(3)
is_train = rng.random(len(t)) < 0.8      # WRONG: random mask
cut = int(0.8 * len(t))                   # RIGHT: chronological cut

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(t[is_train], series[is_train], color='steelblue', s=18, label='train')
axes[0].scatter(t[~is_train], series[~is_train], color='crimson', s=18, label='test')
axes[0].set_title('WRONG: random split\ntrains on future to predict the past = leakage', fontsize=12)
axes[0].set_xlabel('time'); axes[0].set_ylabel('value'); axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

axes[1].plot(t[:cut], series[:cut], color='steelblue', linewidth=2, label='train (earlier)')
axes[1].plot(t[cut:], series[cut:], color='crimson', linewidth=2, label='test (later)')
axes[1].axvline(cut, color='green', linestyle='--', label='split')
axes[1].set_title('RIGHT: chronological split\ntrain on the past, test on the future', fontsize=12)
axes[1].set_xlabel('time'); axes[1].set_ylabel('value'); axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

plt.suptitle('For time-series: never shuffle. Always train on the past, test on the future.',
             fontsize=13)
plt.tight_layout(); plt.show()

print("WHAT YOU'RE SEEING:")
print("  - Left: a random split scatters train and test all through time.")
print("    The model would see future points to predict past ones - impossible in real life.")
print("  - Right: a clean cut - earlier part trains, later part tests.")
print("  - This mimics reality: you only ever have the past to predict the future.")
print("  - So: do NOT use train_test_split(shuffle=True) on time-series.")

## Let's Build a Data Preparation Pipeline

Here's the complete workflow we'll use. In PyTorch, we also need to create a **Dataset** class to handle our data:

In [ ]:
def create_sequences(data: np.ndarray, window_size: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Create input sequences and targets for LSTM.
    
    Parameters:
    -----------
    data : numpy array, shape (n_samples, n_features)
        The time series data
    window_size : int
        Number of time steps in each input sequence
    
    Returns:
    --------
    X : numpy array, shape (n_sequences, window_size, n_features)
        Input sequences
    y : numpy array, shape (n_sequences, n_features)
        Target values (the next value after each sequence)
    """
    X, y = [], []
    
    for i in range(len(data) - window_size):
        # Input: window_size consecutive values
        X.append(data[i : i + window_size])
        # Target: the value right after the window
        y.append(data[i + window_size])
    
    return np.array(X), np.array(y)


class TimeSeriesDataset(Dataset):
    """
    PyTorch Dataset for time-series data.
    
    PyTorch uses Dataset classes to handle data. This is more flexible
    than just passing arrays - you can add data augmentation, lazy loading, etc.
    
    Every Dataset must implement:
    - __len__: returns the number of samples
    - __getitem__: returns one sample by index
    """
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Convert numpy arrays to PyTorch tensors
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self) -> int:
        return len(self.X)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


# Example: Create sequences from sample data
sample_data = np.array([100, 110, 105, 115, 120, 118, 125, 130, 128, 135]).reshape(-1, 1)
window_size = 3

X_sample, y_sample = create_sequences(sample_data, window_size)

print("Original data shape:", sample_data.shape)
print("X shape:", X_sample.shape)
print("y shape:", y_sample.shape)
print("\nExample sequences:")
for i in range(3):
    print(f"  X[{i}]: {X_sample[i].flatten()} → y[{i}]: {y_sample[i].flatten()[0]}")

# Create a PyTorch Dataset
sample_dataset = TimeSeriesDataset(X_sample, y_sample)
print(f"\nPyTorch Dataset created with {len(sample_dataset)} samples")

---

# Part 5: Building LSTM Models in PyTorch

Time to put theory into practice! 

**What's different with PyTorch?** Instead of `model.fit()` doing everything automatically, you write your own training loop. This gives you:
- **More control** over exactly what happens each step
- **Better debugging** - you can inspect anything at any point
- **Flexibility** for custom training schemes

**What's the same?** The concepts from Days 1-2:
- Forward pass → compute predictions
- Loss → measure error
- Backward pass → compute gradients
- Optimizer step → update weights

```
THE PYTORCH PATTERN:

    1. Prepare data (Dataset, DataLoader)
    2. Define model (class inheriting from nn.Module)
    3. Set up loss function and optimizer
    4. Training loop:
       for epoch in range(epochs):
           for batch in dataloader:
               predictions = model(batch)
               loss = criterion(predictions, targets)
               optimizer.zero_grad()
               loss.backward()
               optimizer.step()
    5. Evaluate

    You write the loop yourself - more code, but more control!
```

We'll build our first LSTM with **synthetic data** so we can understand the mechanics with a pattern we control. Then we'll apply it to real power consumption data.

## Creating Synthetic Time-Series Data

Let's create a simple pattern: **sine wave + trend + noise**. This mimics many real-world time series (temperature, stock prices, power consumption).

In [ ]:
# Create synthetic time-series data
# Pattern: sine wave + trend + noise (common in real data!)

np.random.seed(42)

# Time axis
t = np.arange(0, 1000)

# Components:
# 1. Sine wave (cyclical pattern) - period of 50 time steps
sine_wave = np.sin(2 * np.pi * t / 50) * 10

# 2. Linear trend (increasing over time)
trend = t * 0.01

# 3. Random noise
noise = np.random.normal(0, 1, len(t))

# Combined signal
signal = sine_wave + trend + noise

# Reshape to (samples, features)
data = signal.reshape(-1, 1)

# Visualize
plt.figure(figsize=(14, 4))
plt.plot(t[:200], data[:200], 'b-', linewidth=1)
plt.xlabel('Time', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title('Synthetic Time-Series Data\n(Sine wave + Trend + Noise)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Data shape: {data.shape}")
print(f"Data range: {data.min():.2f} to {data.max():.2f}")

In [ ]:
# Step 1: Train/Test Split (temporal - no shuffling!)

train_size = int(len(data) * 0.8)
train_data = data[:train_size]
test_data = data[train_size:]

print(f"Training data: {train_data.shape[0]} samples")
print(f"Test data: {test_data.shape[0]} samples")

# Visualize the split
plt.figure(figsize=(14, 4))
plt.plot(range(train_size), train_data, 'b-', label='Training Data', linewidth=1)
plt.plot(range(train_size, len(data)), test_data, 'r-', label='Test Data', linewidth=1)
plt.axvline(x=train_size, color='green', linestyle='--', label='Train/Test Split')
plt.xlabel('Time', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title('Train/Test Split (Temporal - No Shuffling!)', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Step 2: Normalize the data

scaler = MinMaxScaler(feature_range=(0, 1))

# Fit on training data ONLY
scaler.fit(train_data)

# Transform both train and test
train_scaled = scaler.transform(train_data)
test_scaled = scaler.transform(test_data)

print("After scaling:")
print(f"  Training range: {train_scaled.min():.3f} to {train_scaled.max():.3f}")
print(f"  Test range: {test_scaled.min():.3f} to {test_scaled.max():.3f}")
print(f"\nNote: Test might slightly exceed [0,1] because scaler was fit on training only!")

In [ ]:
# Step 3: Create sequences and PyTorch DataLoaders

WINDOW_SIZE = 20  # Use 20 past values to predict next value
BATCH_SIZE = 32   # Process 32 samples at a time

# Create sequences
X_train, y_train = create_sequences(train_scaled, WINDOW_SIZE)
X_test, y_test = create_sequences(test_scaled, WINDOW_SIZE)

print("Sequence shapes:")
print(f"  X_train: {X_train.shape}  (samples, time_steps, features)")
print(f"  y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_test:  {y_test.shape}")

# Create PyTorch Datasets
train_dataset = TimeSeriesDataset(X_train, y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)

# Create DataLoaders (handle batching and shuffling)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nPyTorch DataLoaders created:")
print(f"  Training batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

# Example: Get one batch
X_batch, y_batch = next(iter(train_loader))
print(f"\nExample batch shapes: X={X_batch.shape}, y={y_batch.shape}")

## Building the LSTM Model in PyTorch

In PyTorch, we define models as **classes** that inherit from `nn.Module`. This is more explicit than Keras but gives you complete control.

In [ ]:
# Define the LSTM Model

class LSTMModel(nn.Module):
    """
    LSTM model for time-series prediction.
    
    In PyTorch, models are classes that inherit from nn.Module.
    You must implement:
    - __init__: define the layers
    - forward: define how data flows through the layers
    """
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, output_size: int):
        super(LSTMModel, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer
        # input_size: number of features at each time step
        # hidden_size: number of LSTM units (memory cells)
        # num_layers: number of stacked LSTM layers
        # batch_first=True: input shape is (batch, seq, features)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        
        # Fully connected layer for final prediction
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the network.
        
        x shape: (batch_size, sequence_length, input_size)
        """
        # Initialize hidden state and cell state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # LSTM forward pass
        # out shape: (batch_size, sequence_length, hidden_size)
        # out contains hidden states for ALL time steps
        out, (hn, cn) = self.lstm(x, (h0, c0))
        
        # We only want the LAST hidden state (for prediction)
        # out[:, -1, :] shape: (batch_size, hidden_size)
        out = self.fc(out[:, -1, :])
        
        return out


# Create model instance
model = LSTMModel(
    input_size=1,      # 1 feature (our single variable)
    hidden_size=50,    # 50 LSTM units (memory cells)
    num_layers=1,      # 1 LSTM layer
    output_size=1      # 1 output (next value prediction)
).to(device)

print("LSTM Model for Time-Series Prediction:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Understanding the Model Architecture

```
MODEL BREAKDOWN:

Input:  (batch_size, 20, 1)
        │          │   │
        │          │   └─ 1 feature (our single variable)
        │          └───── 20 time steps (our window size)
        └───────────── batch of samples
        
        ↓

nn.LSTM(input_size=1, hidden_size=50, num_layers=1):
        • 50 hidden units ("memory cells")
        • Processes all 20 time steps sequentially
        • Outputs: (batch_size, 20, 50) - hidden state at EACH step
        • We take only the LAST step: (batch_size, 50)
        
        Parameters: 
        • 4 * ((input_size + hidden_size) * hidden_size + 2 * hidden_size)
        • 4 * ((1 + 50) * 50 + 2 * 50) = 4 * (2550 + 100) = 10,600
        • The "4" is for the 4 gate weight sets (forget, input, candidate, output)
        • PyTorch keeps TWO bias vectors (input + hidden), hence 2 * hidden_size
        
        ↓

nn.Linear(50, 1):
        • Takes the 50-dimensional hidden state
        • Produces single prediction
        • Parameters: 50 * 1 + 1 = 51
        
        ↓

Output: (batch_size, 1)
        Single predicted value


KEY PyTorch CONCEPTS:

    1. Models inherit from nn.Module
    2. Layers defined in __init__
    3. Forward pass defined in forward()
    4. .to(device) moves model to GPU if available
    5. batch_first=True means input shape is (batch, seq, features)
```

In [ ]:
# Training Loop in PyTorch

# Loss function and optimizer
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training parameters
num_epochs = 50
train_losses = []
val_losses = []

# Split some training data for validation
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

print("Training LSTM...")
print("(Watch the loss decrease!)\n")

for epoch in range(num_epochs):
    # ========== TRAINING ==========
    model.train()  # Set model to training mode
    epoch_train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        # Move data to device (GPU if available)
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Forward pass
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        
        # Backward pass
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute new gradients
        optimizer.step()       # Update weights
        
        epoch_train_loss += loss.item()
    
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # ========== VALIDATION ==========
    model.eval()  # Set model to evaluation mode
    epoch_val_loss = 0.0
    
    with torch.no_grad():  # No gradients needed for validation
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            epoch_val_loss += loss.item()
    
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")

print("\n✅ Training complete!")

In [ ]:
# Plot training history

fig, ax = plt.subplots(1, 1, figsize=(10, 4))

ax.plot(train_losses, label='Training Loss', linewidth=2)
ax.plot(val_losses, label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (MSE)', fontsize=12)
ax.set_title('Training and Validation Loss', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Look for the gap between training and validation loss.")
print("If validation loss starts increasing while training decreases → overfitting!")

In [ ]:
# Make predictions on test data

model.eval()  # Set to evaluation mode

# Get all test data as tensors
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

with torch.no_grad():  # No gradients needed
    predictions_scaled = model(X_test_tensor).cpu().numpy()

# Inverse transform to get original scale
predictions = scaler.inverse_transform(predictions_scaled)
y_test_original = scaler.inverse_transform(y_test)

# Calculate metrics
mse = mean_squared_error(y_test_original, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_original, predictions)
r2 = r2_score(y_test_original, predictions)

print("Test Set Evaluation:")
print("=" * 40)
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  R²:   {r2:.4f}")
print("=" * 40)
print(f"\nR² = {r2:.2%} means the model explains {r2:.0%} of the variance!")

In [ ]:
# Visualize predictions vs actual

plt.figure(figsize=(14, 6))

# Plot actual values
plt.plot(y_test_original, label='Actual', linewidth=1.5, alpha=0.8)

# Plot predictions
plt.plot(predictions, label='Predicted', linewidth=1.5, alpha=0.8)

plt.xlabel('Time Step', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title(f'LSTM Predictions vs Actual Values (R² = {r2:.2%})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

# Zoom in on a section
plt.figure(figsize=(14, 4))
zoom_range = range(50, 100)
plt.plot(zoom_range, y_test_original[zoom_range], 'b-', label='Actual', linewidth=2)
plt.plot(zoom_range, predictions[zoom_range], 'r--', label='Predicted', linewidth=2)
plt.xlabel('Time Step', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title('Zoomed View: LSTM Predictions vs Actual', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## Understanding the Results

```
WHAT WE ACHIEVED:

• The LSTM learned the sine wave pattern!
• It captures the trend (increasing values over time)
• Predictions follow the actual values closely
• R² close to 1 means excellent predictions


KEY METRICS EXPLAINED:

MSE (Mean Squared Error):
    • Average of squared errors
    • Penalizes large errors more heavily
    • Lower is better

RMSE (Root Mean Squared Error):
    • Square root of MSE
    • Same units as the data (interpretable!)
    • "On average, predictions are off by RMSE"

MAE (Mean Absolute Error):
    • Average of absolute errors
    • More robust to outliers than MSE
    • Same units as the data

R² (Coefficient of Determination):
    • Proportion of variance explained by the model
    • 1.0 = perfect predictions
    • 0.0 = no better than predicting the mean
    • Can be negative if predictions are very bad
```

---

---

# Part 6: Real-World Example - Household Power Consumption

You've built an LSTM that works on synthetic data. Now let's tackle **real data** - the same dataset you'll use in your exercise!

This is where things get interesting:
- Real data has **missing values** (sensors fail, data gaps)
- Real data has **multiple patterns** (daily cycles, weekly patterns, seasonal trends)
- Real data needs more careful **preprocessing**

## About the Dataset

The **UCI Household Power Consumption** dataset contains:
- Electric power consumption measurements from one household
- ~2 million rows of minute-by-minute data (Dec 2006 - Nov 2010)
- Features: voltage, current, power consumption, etc.

We'll focus on **Global Active Power** - the total real power consumed.

**Why this dataset?** It's the same one in your exercise, so everything you learn here transfers directly!

---

In [ ]:
# Load the dataset
# The dataset uses ; as separator and has ? for missing values

# URL for the dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip'

# Note: If the URL doesn't work, download manually from:
# https://archive.ics.uci.edu/ml/datasets/Individual+household+electric+power+consumption

try:
    # Try to load from URL
    import urllib.request
    import zipfile
    import io
    
    print("Downloading dataset...")
    with urllib.request.urlopen(url) as response:
        with zipfile.ZipFile(io.BytesIO(response.read())) as z:
            with z.open('household_power_consumption.txt') as f:
                df = pd.read_csv(
                    f, 
                    sep=';', 
                    na_values=['?'],
                    low_memory=False
                )
    print("Download complete!")
except Exception as e:
    print(f"Could not download: {e}")
    print("\nCreating synthetic power data for demonstration...")
    
    # Create synthetic data that mimics power consumption patterns
    np.random.seed(42)
    n_points = 100000
    
    # Create datetime index
    dates = pd.date_range('2007-01-01', periods=n_points, freq='min')
    
    # Create realistic power pattern:
    # - Daily cycle (higher during day)
    # - Weekly cycle (lower on weekends)
    # - Noise
    hour_of_day = dates.hour
    day_of_week = dates.dayofweek
    
    # Base consumption
    base = 1.0
    
    # Daily pattern (higher 7am-10pm)
    daily_pattern = 0.5 * np.sin(2 * np.pi * (hour_of_day - 6) / 24) + 0.5
    
    # Peak hours (morning and evening)
    morning_peak = 0.3 * np.exp(-((hour_of_day - 8) ** 2) / 4)
    evening_peak = 0.5 * np.exp(-((hour_of_day - 19) ** 2) / 6)
    
    # Weekend reduction
    weekend_factor = np.where(day_of_week >= 5, 0.7, 1.0)
    
    # Noise
    noise = np.random.normal(0, 0.2, n_points)
    
    # Combined signal
    power = (base + daily_pattern + morning_peak + evening_peak) * weekend_factor + noise
    power = np.clip(power, 0.1, 5.0)  # Realistic range
    
    df = pd.DataFrame({
        'Date': dates.strftime('%d/%m/%Y'),
        'Time': dates.strftime('%H:%M:%S'),
        'Global_active_power': power,
        'Global_reactive_power': np.random.uniform(0, 0.5, n_points),
        'Voltage': np.random.normal(240, 5, n_points),
        'Global_intensity': power * 4 + np.random.normal(0, 0.5, n_points),
        'Sub_metering_1': np.random.uniform(0, 10, n_points),
        'Sub_metering_2': np.random.uniform(0, 10, n_points),
        'Sub_metering_3': np.random.uniform(0, 20, n_points),
    })
    print("Synthetic data created!")

print(f"\nDataset shape: {df.shape}")
df.head()

In [ ]:
# Explore the data

print("Dataset Info:")
print("=" * 50)
print(df.dtypes)
print("\n" + "=" * 50)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Data Preprocessing

# 1. Handle missing values
# For time-series, we'll use forward fill (use last known value)
# This is better than mean for time-series data

print("Before handling missing values:")
print(f"  Missing in Global_active_power: {df['Global_active_power'].isnull().sum()}")

# Convert to numeric first (handles any string issues)
df['Global_active_power'] = pd.to_numeric(df['Global_active_power'], errors='coerce')

# Forward fill, then backward fill for any remaining at the start
df['Global_active_power'] = df['Global_active_power'].ffill().bfill()

print("After handling missing values:")
print(f"  Missing in Global_active_power: {df['Global_active_power'].isnull().sum()}")

In [ ]:
# Create datetime index

# Combine Date and Time columns
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.set_index('datetime', inplace=True)

print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Frequency: approximately every minute")

In [ ]:
# Resample to hourly (minute-level is too granular for learning patterns)
# This also reduces data size for faster training

df_hourly = df[['Global_active_power']].resample('h').mean()

print(f"Original data: {len(df)} rows (minute-level)")
print(f"Resampled data: {len(df_hourly)} rows (hourly)")
print(f"\nHourly statistics:")
print(df_hourly.describe())

In [ ]:
# Visualize the data

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Full time series
axes[0].plot(df_hourly.index, df_hourly['Global_active_power'], linewidth=0.5)
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Global Active Power (kW)', fontsize=12)
axes[0].set_title('Household Power Consumption Over Time', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Plot 2: One week of data to see patterns
one_week = df_hourly.iloc[:24*7]
axes[1].plot(one_week.index, one_week['Global_active_power'], linewidth=1.5)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Global Active Power (kW)', fontsize=12)
axes[1].set_title('Power Consumption - One Week (See Daily Patterns!)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice the daily patterns - higher during day, lower at night!")
print("This is what the LSTM will learn to predict.")

In [ ]:
# Prepare data for LSTM

# Get values as numpy array
power_data = df_hourly['Global_active_power'].values.reshape(-1, 1)

# Remove any NaN that might have appeared during resampling
power_data = power_data[~np.isnan(power_data).any(axis=1)]

# Train/test split (temporal)
train_size = int(len(power_data) * 0.8)
train_power = power_data[:train_size]
test_power = power_data[train_size:]

print(f"Training samples: {len(train_power)}")
print(f"Test samples: {len(test_power)}")

# Normalize
scaler_power = MinMaxScaler(feature_range=(0, 1))
train_power_scaled = scaler_power.fit_transform(train_power)
test_power_scaled = scaler_power.transform(test_power)

# Create sequences
# Use 24 hours (1 day) to predict next hour
WINDOW_SIZE_POWER = 24
BATCH_SIZE_POWER = 64

X_train_power, y_train_power = create_sequences(train_power_scaled, WINDOW_SIZE_POWER)
X_test_power, y_test_power = create_sequences(test_power_scaled, WINDOW_SIZE_POWER)

# Create PyTorch Datasets and DataLoaders
train_dataset_power = TimeSeriesDataset(X_train_power, y_train_power)
test_dataset_power = TimeSeriesDataset(X_test_power, y_test_power)

# Split training into train/validation
val_size_power = int(0.1 * len(train_dataset_power))
train_size_power = len(train_dataset_power) - val_size_power
train_subset_power, val_subset_power = torch.utils.data.random_split(
    train_dataset_power, [train_size_power, val_size_power]
)

train_loader_power = DataLoader(train_subset_power, batch_size=BATCH_SIZE_POWER, shuffle=True)
val_loader_power = DataLoader(val_subset_power, batch_size=BATCH_SIZE_POWER, shuffle=False)
test_loader_power = DataLoader(test_dataset_power, batch_size=BATCH_SIZE_POWER, shuffle=False)

print(f"\nAfter creating sequences:")
print(f"  X_train: {X_train_power.shape}")
print(f"  y_train: {y_train_power.shape}")
print(f"  X_test: {X_test_power.shape}")
print(f"  y_test: {y_test_power.shape}")
print(f"\nDataLoaders: {len(train_loader_power)} train batches, {len(val_loader_power)} val batches")

In [ ]:
# Build Stacked LSTM model for power prediction

class StackedLSTMModel(nn.Module):
    """
    Stacked LSTM model with multiple layers for more complex patterns.
    
    First layer learns low-level patterns (hourly fluctuations)
    Second layer learns high-level patterns (daily cycles)
    """
    def __init__(self, input_size: int, hidden_sizes: list[int], output_size: int, dropout: float = 0.2):
        super(StackedLSTMModel, self).__init__()
        
        self.hidden_sizes = hidden_sizes
        
        # First LSTM layer
        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_sizes[0],
            num_layers=1,
            batch_first=True
        )
        
        # Second LSTM layer
        self.lstm2 = nn.LSTM(
            input_size=hidden_sizes[0],
            hidden_size=hidden_sizes[1],
            num_layers=1,
            batch_first=True
        )
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_sizes[1], 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, output_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # First LSTM layer (returns sequence for next layer)
        h0_1 = torch.zeros(1, x.size(0), self.hidden_sizes[0]).to(x.device)
        c0_1 = torch.zeros(1, x.size(0), self.hidden_sizes[0]).to(x.device)
        out, _ = self.lstm1(x, (h0_1, c0_1))
        
        # Second LSTM layer
        h0_2 = torch.zeros(1, x.size(0), self.hidden_sizes[1]).to(x.device)
        c0_2 = torch.zeros(1, x.size(0), self.hidden_sizes[1]).to(x.device)
        out, _ = self.lstm2(out, (h0_2, c0_2))
        
        # Take last time step, apply dropout
        out = self.dropout(out[:, -1, :])
        
        # Fully connected layers
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        
        return out


# Create model
model_power = StackedLSTMModel(
    input_size=1,
    hidden_sizes=[64, 32],  # Two LSTM layers with 64 and 32 units
    output_size=1,
    dropout=0.2
).to(device)

print("Stacked LSTM Model for Power Consumption:")
print(model_power)

total_params = sum(p.numel() for p in model_power.parameters())
print(f"\nTotal parameters: {total_params:,}")

## Understanding Stacked LSTMs

Remember from **Day 2** how CNNs build hierarchies?

```
CNN HIERARCHY (Day 2):

    Conv Layer 1: Detects EDGES (low-level patterns)
           ↓
    Conv Layer 2: Combines edges into SHAPES
           ↓  
    Conv Layer 3: Combines shapes into OBJECTS (high-level patterns)


LSTM HIERARCHY (Today):

    LSTM Layer 1: Detects LOW-LEVEL temporal patterns
                  (hourly fluctuations, immediate trends)
           ↓
    LSTM Layer 2: Combines into HIGH-LEVEL patterns
                  (daily cycles, weekly trends)

    
    Same principle: Each layer learns more ABSTRACT patterns!


WHY STACK LSTM LAYERS?

Single LSTM:
    • Learns one level of patterns
    • Good for simple sequences
    
Stacked LSTMs:
    • First layer: learns low-level patterns (hour-to-hour changes)
    • Second layer: learns high-level patterns (day-to-day cycles)
    • Just like CNNs: simple → complex!


OUR PYTORCH ARCHITECTURE:

    Input: 24 hours of power consumption
    
    │ ← shape: (batch, 24, 1)
    ▼
    ┌────────────────────────────────────────┐
    │ nn.LSTM(input=1, hidden=64)            │
    │                                        │
    │ Processes all 24 time steps            │
    │ Output: (batch, 24, 64) - ALL steps    │
    │ → Learns: "what patterns in each hour?"│
    └────────────────────────────────────────┘
    │ ← Full sequence passed to next layer
    ▼
    ┌────────────────────────────────────────┐
    │ nn.LSTM(input=64, hidden=32)           │
    │                                        │
    │ Processes the 24 "summaries" from L1   │
    │ Output: (batch, 24, 32)                │
    │ → Take LAST step: (batch, 32)          │
    │ → Learns: "what's the overall pattern?"│
    └────────────────────────────────────────┘
    │ ← shape: (batch, 32)
    ▼
    ┌────────────────────────────────────────┐
    │ nn.Dropout(0.2)                        │
    │ → Regularization to prevent overfitting│
    └────────────────────────────────────────┘
    │
    ▼
    ┌────────────────────────────────────────┐
    │ nn.Linear(32, 16) → ReLU               │
    │ nn.Linear(16, 1)                       │
    │                                        │
    │ Maps hidden state to prediction        │
    └────────────────────────────────────────┘
    │ ← shape: (batch, 1)
    ▼
    Prediction: next hour's power consumption


KEY PyTorch DIFFERENCE FROM KERAS:

    Keras: return_sequences=True/False controls output
    
    PyTorch: LSTM always returns full sequence
             You select which outputs to use:
             out[:, -1, :]  → last time step only
             out           → all time steps
```

In [ ]:
# Train the power consumption model with early stopping

criterion_power = nn.MSELoss()
optimizer_power = torch.optim.Adam(model_power.parameters(), lr=0.001)

num_epochs_power = 100
train_losses_power = []
val_losses_power = []

# Early stopping parameters
patience = 10
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print("Training Power Consumption Model...")
print("(Early stopping will stop when validation loss stops improving)\n")

for epoch in range(num_epochs_power):
    # ========== TRAINING ==========
    model_power.train()
    epoch_train_loss = 0.0
    
    for X_batch, y_batch in train_loader_power:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        predictions = model_power(X_batch)
        loss = criterion_power(predictions, y_batch)
        
        optimizer_power.zero_grad()
        loss.backward()
        optimizer_power.step()
        
        epoch_train_loss += loss.item()
    
    avg_train_loss = epoch_train_loss / len(train_loader_power)
    train_losses_power.append(avg_train_loss)
    
    # ========== VALIDATION ==========
    model_power.eval()
    epoch_val_loss = 0.0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader_power:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            predictions = model_power(X_batch)
            loss = criterion_power(predictions, y_batch)
            epoch_val_loss += loss.item()
    
    avg_val_loss = epoch_val_loss / len(val_loader_power)
    val_losses_power.append(avg_val_loss)
    
    # ========== EARLY STOPPING ==========
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = model_power.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch + 1}")
            break
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_power}], Train: {avg_train_loss:.6f}, Val: {avg_val_loss:.6f}")

# Restore best model
if best_model_state is not None:
    model_power.load_state_dict(best_model_state)

print(f"\n✅ Training complete! Best val loss: {best_val_loss:.6f}")

In [ ]:
# Evaluate the power consumption model

model_power.eval()

# Get predictions on test set
X_test_power_tensor = torch.tensor(X_test_power, dtype=torch.float32).to(device)

with torch.no_grad():
    predictions_power_scaled = model_power(X_test_power_tensor).cpu().numpy()

# Inverse transform
predictions_power = scaler_power.inverse_transform(predictions_power_scaled)
y_test_power_original = scaler_power.inverse_transform(y_test_power)

# Metrics
mse_power = mean_squared_error(y_test_power_original, predictions_power)
rmse_power = np.sqrt(mse_power)
mae_power = mean_absolute_error(y_test_power_original, predictions_power)
r2_power = r2_score(y_test_power_original, predictions_power)

print("Power Consumption Model - Test Evaluation:")
print("=" * 50)
print(f"  RMSE: {rmse_power:.4f} kW")
print(f"  MAE:  {mae_power:.4f} kW")
print(f"  R²:   {r2_power:.4f}")
print("=" * 50)

# Plot training history
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(train_losses_power, label='Training Loss', linewidth=2)
ax.plot(val_losses_power, label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (MSE)', fontsize=12)
ax.set_title('Power Model Training History', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Full test set
axes[0].plot(y_test_power, label='Actual', linewidth=1, alpha=0.8)
axes[0].plot(predictions_power, label='Predicted', linewidth=1, alpha=0.8)
axes[0].set_xlabel('Time Step', fontsize=12)
axes[0].set_ylabel('Global Active Power (kW)', fontsize=12)
axes[0].set_title(f'Power Consumption: Actual vs Predicted (R² = {r2_power:.2%})', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# One week zoom
week_range = range(0, min(24*7, len(y_test_power)))
axes[1].plot(week_range, y_test_power[week_range], 'b-', label='Actual', linewidth=1.5)
axes[1].plot(week_range, predictions_power[week_range], 'r--', label='Predicted', linewidth=1.5)
axes[1].set_xlabel('Hour', fontsize=12)
axes[1].set_ylabel('Global Active Power (kW)', fontsize=12)
axes[1].set_title('One Week Zoom: Model Captures Daily Patterns!', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The LSTM learned the daily consumption patterns!")
print("Notice how it captures the daily peaks and troughs.")

---

# Part 7: Keras Comparison (For Reference)

You've now learned PyTorch! But you might see Keras code elsewhere (like in Day 2), so here's a quick comparison to help you translate between them.

## PyTorch vs Keras: Side by Side

```
THE TWO FRAMEWORKS:

    PYTORCH (what we used today):
    ├── Industry standard for RESEARCH
    ├── More explicit control
    ├── You write the training loop
    ├── Used by: Meta, Tesla, OpenAI, most papers
    └── Great for: Understanding, customizing, research

    KERAS / TensorFlow (what you used in Day 2):
    ├── High-level, easy to use
    ├── model.fit() does everything automatically
    ├── Less code for standard tasks
    ├── Used by: Google, many production systems
    └── Great for: Quick prototyping, standard architectures


SAME CONCEPTS, DIFFERENT SYNTAX:

┌────────────────────┬──────────────────────────┬──────────────────────────┐
│ Concept            │ PyTorch                  │ Keras                    │
├────────────────────┼──────────────────────────┼──────────────────────────┤
│ Import             │ import torch.nn as nn    │ from tensorflow import   │
│                    │                          │   keras                  │
├────────────────────┼──────────────────────────┼──────────────────────────┤
│ Define model       │ class Model(nn.Module):  │ model = keras.Sequential │
│                    │     def __init__():      │   ([layers...])          │
│                    │     def forward():       │                          │
├────────────────────┼──────────────────────────┼──────────────────────────┤
│ LSTM layer         │ nn.LSTM(input, hidden,   │ layers.LSTM(units,       │
│                    │   batch_first=True)      │   return_sequences=...)  │
├────────────────────┼──────────────────────────┼──────────────────────────┤
│ Dense layer        │ nn.Linear(in, out)       │ layers.Dense(units)      │
├────────────────────┼──────────────────────────┼──────────────────────────┤
│ Training           │ for epoch:               │ model.compile(...)       │
│                    │   for batch:             │ model.fit(X, y,          │
│                    │     pred = model(X)      │   epochs=...,            │
│                    │     loss = criterion()   │   batch_size=...)        │
│                    │     loss.backward()      │                          │
│                    │     optimizer.step()     │                          │
├────────────────────┼──────────────────────────┼──────────────────────────┤
│ Evaluation         │ model.eval()             │ model.evaluate(X, y)     │
│                    │ with torch.no_grad():    │                          │
│                    │   pred = model(X)        │                          │
└────────────────────┴──────────────────────────┴──────────────────────────┘


WHY WE CHOSE PYTORCH TODAY:

    1. Industry relevance - Most ML jobs require PyTorch
    2. Understanding - Writing the loop helps you understand training
    3. Your exercises use PyTorch - Daily challenge is in PyTorch
    4. Research skills - Reading papers requires PyTorch knowledge
    5. Flexibility - Custom training schemes are easier in PyTorch
```

---

## If You Need to Use Keras Instead

Here's the same LSTM in Keras for reference (like Day 2 style):

```python
# ============================================================
# KERAS VERSION (for reference only)
# ============================================================

from tensorflow import keras
from tensorflow.keras import layers

# Build model - much less code!
model = keras.Sequential([
    layers.Input(shape=(24, 1)),  # (time_steps, features)
    layers.LSTM(64, return_sequences=True),  # return full sequence
    layers.LSTM(32),  # return only last step
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

# Compile
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train - one line!
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=10)]
)

# Evaluate
predictions = model.predict(X_test)
```

```
KEY DIFFERENCES:

    Keras:
    ├── Model defined with Sequential([layers])
    ├── Training is ONE LINE: model.fit()
    ├── return_sequences controls LSTM output
    └── ~15 lines of code

    PyTorch (what we learned):
    ├── Model defined as a CLASS
    ├── Training loop is EXPLICIT (for epoch, for batch...)
    ├── You slice output: out[:, -1, :]
    └── ~50 lines of code

    SAME RESULT, different trade-offs:
    • Keras: Faster to write, less control
    • PyTorch: More code, full control
```

---

---

# Summary: Your Deep Learning Journey

Let's step back and see how far you've come:

```
YOUR COMPLETE DEEP LEARNING TOOLKIT:

┌─────────────────────────────────────────────────────────────────────┐
│  DAY 1:   THE FOUNDATIONS                                          │
├─────────────────────────────────────────────────────────────────────┤
│  • Perceptrons: Single neurons that learn                          │
│  • Activations: ReLU, sigmoid, tanh - why we need non-linearity   │
│  • Dense layers: Every input connects to every neuron              │
│  • Loss functions: How to measure "wrongness"                      │
│  • Optimizers: SGD, Adam - how to learn from mistakes              │
│  • Overfitting: Training too well on training data                 │
│                                                                     │
│  KEY INSIGHT: Neural networks learn by adjusting weights           │
│               based on how wrong their predictions are.            │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  DAY 2: SPATIAL PATTERNS (CNNs) - Keras                            │
├─────────────────────────────────────────────────────────────────────┤
│  • Problem: Dense networks lose spatial relationships               │
│  • Solution: Conv2D filters that slide across images               │
│  • Weight sharing: Same filter at every position                   │
│  • Hierarchy: Edges → Shapes → Objects                              │
│                                                                     │
│  KEY INSIGHT: Different data structures need different             │
│               architectures. CNNs preserve SPATIAL structure.       │
└─────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│  DAY 3: TEMPORAL PATTERNS (RNNs/LSTMs) - PyTorch - TODAY!          │
├─────────────────────────────────────────────────────────────────────┤
│  • Problem: Dense networks lose temporal relationships              │
│  • Solution: RNNs that process sequences step-by-step              │
│  • Hidden state: "Memory" that carries information across time     │
│  • Weight sharing: Same operation at every time step               │
│  • LSTMs: Gates that control what to remember/forget               │
│  • Hierarchy: Short-term patterns → Long-term patterns              │
│  • NEW: PyTorch framework - explicit training loops!               │
│                                                                     │
│  KEY INSIGHT: Just like CNNs for space, RNNs preserve TEMPORAL     │
│               structure. Same principle, different dimension!       │
└─────────────────────────────────────────────────────────────────────┘


THE PATTERN YOU'VE LEARNED:

    1. Identify the STRUCTURE in your data
       • Tabular data → Dense networks
       • Spatial data (images) → CNNs
       • Sequential data (time-series) → RNNs/LSTMs

    2. Choose architecture that PRESERVES that structure
       • Weight sharing is key (filters for space, recurrence for time)

    3. Same training principles apply everywhere
       • Forward pass, loss, backward pass, optimizer step
       • Epochs, batches, validation, early stopping
       • Watch for overfitting!


YOUR NEW PYTORCH SKILLS:

    • Define models as classes (nn.Module)
    • Write explicit training loops
    • Use Dataset and DataLoader for batching
    • model.train() vs model.eval()
    • torch.no_grad() for inference
    • Move data to GPU with .to(device)
```

---

## Connecting to Your Exercises

You have two exercises. Here's exactly how this lecture prepares you:

```
┌─────────────────────────────────────────────────────────────────────┐
│  MAIN EXERCISE: Household Power Consumption LSTM                   │
│  Difficulty: ★★★☆☆ (You've practiced this!)                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  The exercise may use Keras or PyTorch. Either way, you're ready! │
│                                                                     │
│  Exercise Part → What You Learned Today                             │
│  ─────────────────────────────────────────────────                  │
│                                                                     │
│  Part 1: Data Import                                                │
│     → We did this in Part 6! (pandas, read_csv, head)              │
│                                                                     │
│  Part 2: Handle Missing Values                                      │
│     → We used ffill() in Part 6                                    │
│     → You can also use fillna(df.mean())                           │
│                                                                     │
│  Part 3: Data Visualization                                         │
│     → We used resample('h').mean() and matplotlib                  │
│     → Try resample('D') for daily aggregation                      │
│                                                                     │
│  Part 4: Preprocessing for LSTM                                     │
│     → MinMaxScaler: Part 4 (fit on train only!)                    │
│     → create_sequences(): Part 4 (copy our function!)              │
│     → Temporal split: Part 4 (no shuffling!)                       │
│     → PyTorch: TimeSeriesDataset and DataLoader                    │
│                                                                     │
│  Part 5: Build LSTM Model                                           │
│     → PyTorch: class LSTMModel(nn.Module): Part 5                  │
│     → Or Keras: keras.Sequential([...]) - see Part 7               │
│                                                                     │
│  Part 6: Train and Evaluate                                         │
│     → PyTorch: Training loop with optimizer.step()                 │
│     → Plot train/val losses, calculate R²                          │
│                                                                     │
│  YOU ARE FULLY PREPARED!                                            │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│  DAILY CHALLENGE: Stock Price Prediction (PyTorch)                 │
│  Difficulty: ★★★★☆ (You've got this - same pattern!)              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  This uses EXACTLY what we learned today! Your roadmap:            │
│                                                                     │
│  1. Data prep: Same as power example                                │
│     • Load CSV with pandas                                         │
│     • Normalize with MinMaxScaler                                  │
│     • create_sequences()                                           │
│     • TimeSeriesDataset and DataLoader                             │
│                                                                     │
│  2. Model: Copy LSTMModel from Part 5                               │
│     class LSTMModel(nn.Module):                                    │
│         def __init__(self): ...                                    │
│         def forward(self, x): ...                                  │
│                                                                     │
│  3. Training loop: Copy from Part 5                                 │
│     for epoch in range(epochs):                                    │
│         model.train()                                              │
│         for X_batch, y_batch in train_loader:                      │
│             predictions = model(X_batch)                           │
│             loss = criterion(predictions, y_batch)                 │
│             optimizer.zero_grad()                                  │
│             loss.backward()                                        │
│             optimizer.step()                                       │
│                                                                     │
│  4. Evaluation:                                                     │
│     model.eval()                                                   │
│     with torch.no_grad():                                          │
│         predictions = model(X_test_tensor)                         │
│     r2 = r2_score(y_test, predictions.numpy())                     │
│                                                                     │
│  TIPS:                                                             │
│  • The lecture code is almost exactly what you need!              │
│  • Don't forget model.train() and model.eval()                    │
│  • Move data to device: X_batch.to(device)                        │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## Quick Reference Code

Here's the essential code you'll need for your exercises:

In [ ]:
# ============================================================
# QUICK REFERENCE: PyTorch LSTM for Time-Series
# ============================================================

# 1. Data Preparation
# -------------------
# from sklearn.preprocessing import MinMaxScaler
# 
# scaler = MinMaxScaler()
# data_scaled = scaler.fit_transform(data)
# 
# X, y = create_sequences(data_scaled, window_size=24)
# 
# dataset = TimeSeriesDataset(X, y)
# train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 2. Define Model
# ---------------
# class LSTMModel(nn.Module):
#     def __init__(self, input_size, hidden_size, output_size):
#         super().__init__()
#         self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
#         self.fc = nn.Linear(hidden_size, output_size)
#         self.hidden_size = hidden_size
#     
#     def forward(self, x):
#         h0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
#         c0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
#         out, _ = self.lstm(x, (h0, c0))
#         out = self.fc(out[:, -1, :])  # Last time step
#         return out
# 
# model = LSTMModel(1, 50, 1).to(device)

# 3. Training Loop
# ----------------
# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# 
# for epoch in range(epochs):
#     model.train()
#     for X_batch, y_batch in train_loader:
#         X_batch = X_batch.to(device)
#         y_batch = y_batch.to(device)
#         
#         predictions = model(X_batch)
#         loss = criterion(predictions, y_batch)
#         
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

# 4. Evaluate
# -----------
# model.eval()
# with torch.no_grad():
#     X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
#     predictions = model(X_test_tensor).cpu().numpy()
# 
# predictions_original = scaler.inverse_transform(predictions)
# r2 = r2_score(y_test_original, predictions_original)

print("Quick reference code is in the comments above!")
print("Copy and modify for your exercises.")

---

# You're Ready!

Let's reflect on what you've accomplished in just 3 days:

```
DAY 1:   "What's a neural network, and how does it learn?"
         → You learned neurons, activations, loss, optimizers, training

DAY 2:   "How do we handle images?"
         → You built CNNs that see patterns in space

DAY 3:   "How do we handle sequences?"
(Today!) → You built LSTMs that find patterns in time


You now understand the THREE PILLARS of deep learning:

    1. Dense Networks  → For tabular/flat data
    2. CNNs            → For spatial data (images)
    3. RNNs/LSTMs      → For sequential data (time-series, text)

This covers the vast majority of real-world deep learning applications!
```

## Before You Go: Quick Checklist

For the **main exercise** (Keras LSTM), make sure you can:
- [ ] Load and explore time-series data with pandas
- [ ] Handle missing values (fillna or ffill)
- [ ] Normalize data with MinMaxScaler (fit on train only!)
- [ ] Create sequences with the windowing function
- [ ] Build LSTM model with keras.Sequential
- [ ] Train and evaluate, plot the learning curves

For the **daily challenge** (PyTorch), make sure you understand:
- [ ] How to convert numpy to torch tensors
- [ ] The Dataset and DataLoader pattern
- [ ] How to define a model as nn.Module
- [ ] The training loop: forward → loss → backward → step

## Final Tips

1. **Start with the Keras exercise** - it uses concepts you know well
2. **Copy code from this lecture** - that's what it's here for!
3. **Don't memorize** - understand the patterns, look up syntax
4. **If stuck on PyTorch** - do Keras first, concepts transfer directly
5. **Experiment!** - try different window sizes, LSTM units, layers

---

**You've learned a lot. Now go build something!**

---